[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Why sqlite3 &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell writes the year of readings and loads it into `scratch/stations.db`, as the notebook's
worked examples did. Run it first, then the tasks in order, since task 6 copies the database that
task 5 adds a table to. The last cell removes the scratch folder.


In [1]:
import csv
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
READINGS = SCRATCH / "readings.csv"
DATABASE = SCRATCH / "stations.db"
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}

with open(READINGS, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["station", "hour", "celsius"])
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = ""
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            writer.writerow([station, hour.strftime("%Y-%m-%dT%H:%M"), celsius])


def reading(row):
    """One line of the CSV as the values of one row of readings. An empty reading becomes None."""
    celsius = float(row["celsius"]) if row["celsius"] else None
    return row["station"], row["hour"], celsius


conn = sqlite3.connect(DATABASE)
conn.execute("CREATE TABLE readings (station TEXT NOT NULL, hour TEXT NOT NULL, celsius REAL)")
with open(READINGS, newline="", encoding="utf-8") as file:
    conn.executemany("INSERT INTO readings (station, hour, celsius) VALUES (?, ?, ?)",
                     map(reading, csv.DictReader(file)))
conn.commit()
print("rows loaded:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
conn.close()


rows loaded: 35040


**1.** The warmest hour at Bergen.


In [2]:
conn = sqlite3.connect(DATABASE)
warmest = conn.execute("""
    SELECT hour, celsius FROM readings
    WHERE station = ? AND celsius IS NOT NULL
    ORDER BY celsius DESC, hour
    LIMIT 1
""", ("Bergen",)).fetchone()
conn.close()

print("warmest hour at Bergen:", warmest)


warmest hour at Bergen: ('2025-07-21T15:00', 20.8)


`DESC` reverses the sort, so the highest reading comes first. `NULL` sorts before every number going
up, and so after every number coming down, which means this query would find a real reading even
without `IS NOT NULL`. The condition stays, because it says what the query means.


**2.** Tromso in July.


In [3]:
conn = sqlite3.connect(DATABASE)
(mean,) = conn.execute(
    "SELECT AVG(celsius) FROM readings WHERE station = ? AND hour LIKE ?", ("Tromso", "2025-07%")
).fetchone()
conn.close()

print("Tromso in July 2025:", round(mean, 1))


Tromso in July 2025: 12.4


`LIKE '2025-07%'` matches every hour that begins with that year and month, since `%` stands for any
text. The pattern goes in through a placeholder like any other value.


**3.** Hours with no reading.


In [4]:
conn = sqlite3.connect(DATABASE)
missing = conn.execute("""
    SELECT station, COUNT(*) - COUNT(celsius)
    FROM readings
    GROUP BY station
    ORDER BY station
""").fetchall()
conn.close()

print(missing)


[('Bergen', 0), ('Oslo', 0), ('Svalbard', 24), ('Tromso', 0)]


`COUNT(*)` counts every row and `COUNT(celsius)` only the rows with a reading, so the difference is
the number of hours with none.


**4.** Readings below freezing in 2025.


In [5]:
conn = sqlite3.connect(DATABASE)
freezing = conn.execute("""
    SELECT station, COUNT(*)
    FROM readings
    WHERE celsius < 0 AND hour LIKE ?
    GROUP BY station
    ORDER BY station
""", ("2025%",)).fetchall()
conn.close()

print(freezing)


[('Bergen', 1220), ('Oslo', 1848), ('Svalbard', 5871), ('Tromso', 3203)]


A missing reading is not below freezing: `NULL < 0` is neither true nor false, so `WHERE` leaves
those rows out without being told to.


**5.** A table of stations.


In [6]:
conn = sqlite3.connect(DATABASE)
conn.execute("CREATE TABLE stations (name TEXT NOT NULL, latitude REAL NOT NULL)")
conn.executemany("INSERT INTO stations (name, latitude) VALUES (?, ?)",
                 [("Bergen", 60.39), ("Oslo", 59.91), ("Svalbard", 78.22), ("Tromso", 69.65)])
conn.commit()
conn.close()

check = sqlite3.connect(DATABASE)
print(check.execute("SELECT name, latitude FROM stations ORDER BY name").fetchall())
check.close()


[('Bergen', 60.39), ('Oslo', 59.91), ('Svalbard', 78.22), ('Tromso', 69.65)]


The new connection finds the rows because `commit` ran before the first connection closed. Without
it, the table would exist and hold nothing.


**6.** A copy, checked.


In [7]:
copy = SCRATCH / "task-copy.db"
shutil.copy(DATABASE, copy)

original, duplicate = sqlite3.connect(DATABASE), sqlite3.connect(copy)
count = "SELECT COUNT(*) FROM readings"
print("begins with the header:", copy.read_bytes()[:16] == b"SQLite format 3\x00")
print("as many rows as the original:", duplicate.execute(count).fetchone() == original.execute(count).fetchone())
original.close()
duplicate.close()


begins with the header: True
as many rows as the original: True


Every SQLite database file begins with the same 16 bytes, the text `SQLite format 3` and a zero byte,
which is how a program can tell a database from a CSV before opening it.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Why sqlite3](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/01-why-sqlite3.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
